# IITM Reinforcement Learning Course Project — Final Notebook

**Course:** IITM Web M.Tech Program — Course ID 6002W  
**Project:** Industrial Inventory Control using Reinforcement Learning  
**Roll number:** DA25M579  
**Assigned parameter variant:** V030

This notebook is the final reproducibility and evidence record for the five distinct techniques selected for the leaderboard.

## 1. Course requirements covered

The notebook records the assigned configuration, common state preprocessing and action mapping, technique-specific representations and algorithms, important hyperparameters, reward shaping, training/local validation, convergence evidence, the five-technique comparison, final export mapping, and policy validation steps.

The five distinct techniques are **PPO, DQN, Neural Network SARSA, A2C, and Double DQN**.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
TRAINING_ROOT = PROJECT_ROOT / 'training_pipelines'
for path in (PROJECT_ROOT, TRAINING_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

config = json.loads((TRAINING_ROOT / 'assigned_config.json').read_text(encoding='utf-8'))
pd.Series({
    'roll_number': config['roll_number'],
    'variant_id': config['variant_id'],
    'config_fingerprint': config['config_fingerprint'],
})

## 2. Assigned configuration and official environment

The frozen assigned configuration is V030. The official observation contains current inventory, the four-day arrival pipeline, seven-day demand history, day index and capacity utilisation. The official environment, cost function and leaderboard evaluator are not modified.

The public interface is `run_policy(observation) -> [q1, q2, q3]`, where each quantity is one of `{0,10,...,100}`. Policies receive only the documented observation and perform inference only.

In [ ]:
from industrial_inventory_env import IndustrialInventoryEnv
env = IndustrialInventoryEnv(student_config=config, scenario_mode='random', domain_randomization=True)
obs, info = env.reset(seed=900)
print('Observation keys:', sorted(obs))
print('Action space:', env.action_space)
print('Inventory shape:', np.asarray(obs['inventory']).shape)
print('Arrival pipeline shape:', np.asarray(obs['arrival_pipeline']).shape)
print('Demand history shape:', np.asarray(obs['demand_history']).shape)
env.close()

## 3. State representations and action mapping

**PPO:** 76-D Representation B. **DQN:** 35-D hand-engineered features. **Neural Network SARSA:** 76-D Representation B. **A2C:** 99-D Representation C. **Double DQN:** 35-D hand-engineered features.

Representation B is 38 normalized raw features plus 38 engineered inventory-control features. Representation C extends Representation B with EWMA demand, demand slope, demand CV, recent extrema, lead-time demand uncertainty, safety-stock signals and weekly phase features.

DQN-family joint-action methods use the full 11 x 11 x 11 = 1,331 action catalogue internally and decode the selected joint action to three order quantities.

In [ ]:
from training_pipelines.src.features.observation import RAW_FEATURE_DIM
from training_pipelines.src.features.engineered import REPRESENTATION_B_DIM
from training_pipelines.src.environment.action_codec import JOINT_ACTION_SIZE
print('Raw dimension:', RAW_FEATURE_DIM)
print('Representation B dimension:', REPRESENTATION_B_DIM)
print('Representation C dimension:', 99)
print('Joint action count:', JOINT_ACTION_SIZE)
print('External quantities:', list(range(0, 101, 10)))

## 4. Reward and validation methodology

The official reward is negative daily total cost divided by 100. Training-time `ShapedReward`, where used, is annealed to zero and is not used to report final cost.

The official-style local holdout is 40 seeds (900–939) across five scenario modes — stationary, seasonal, trend, shock and random — for 200 episodes. The course policy validator is also run against every final policy file.

## 5. Reproducible final training pipelines

These are the final recipes represented by the retained training source. They are shown as commands so the notebook remains a reproducibility record without starting long training jobs automatically.

In [ ]:
# PPO + Representation B
# python training_pipelines/training_scripts/train_ppo_rep_b_experiment.py --timesteps 500000 --learning-rate 0.0006 --entropy-coef 0.0 --seed 20260727 --device cpu --run-name final_ppo_rep_b_500k

# DQN
# python training_pipelines/training_scripts/train_dqn.py --timesteps 150000 --seed 20260727 --run-name final_dqn_150k

# Neural Network SARSA
# python training_pipelines/training_scripts/train_neural_sarsa_final.py --total-transitions 500000 --seed 20260825 --run-name final_neural_sarsa_500k

# A2C + Representation C
# python training_pipelines/training_scripts/train_a2c_rep_c.py --timesteps 500000 --n-envs 4 --seed 20260902 --learning-rate 0.0005 --gamma 0.99 --gae-lambda 0.95 --ent-coef 0.001 --n-steps 64 --vf-coef 0.5 --run-name final_a2c_rep_c_500k

# Double DQN
# python training_pipelines/training_scripts/train_double_dqn_experiment.py --timesteps 100000 --seed 20260727 --run-name final_double_dqn_100k

## 6. Final public leaderboard comparison

Lower cost is better. The A2C slot was replaced by the Representation C candidate after an official 200-episode local mean of **94,180.86** and a public score of **94,660.12**. The final public Top-5 average is **99,566.57**.

In [ ]:
public_results = pd.DataFrame({
    'Technique': ['PPO','DQN','A2C + Representation C','Neural Network SARSA','Double DQN'],
    'Public mean cost': [83610.75, 90670.62, 94660.12, 113072.38, 115819.00],
    'Status': ['Retained','Retained','Final A2C replacement','Retained public artifact','Retained public artifact'],
})
display(public_results)
print(f"Final public Top-5 average: {public_results['Public mean cost'].mean():,.2f}")

## 7. Final local validation evidence

Final local holdout evidence used for selection is summarised below.

In [ ]:
local_results = pd.DataFrame({
    'Technique': ['PPO + Rep-B','DQN','A2C + Rep-C','Neural Network SARSA'],
    'Official holdout mean cost': [83039.20, 96871.30, 94180.86, 111730.60],
    'Std': [9777.14, 11858.00, 7961.56, 9941.00],
    'Mean service': [0.992973, 0.988895, 0.994717, 0.992882],
})
display(local_results)

### A2C Representation C holdout

Scenario means: random 94,911.88; seasonal 92,426.00; shock 95,611.69; stationary 92,332.50; trend 95,622.25. Overall mean cost 94,180.86; mean service 0.994717.

### SARSA final gate

The last targeted SARSA refinement screened at 124,306.75 and scored 117,164.21 on the official 200-episode holdout, so the retained 113,072.38 public artifact was kept.

### Double DQN final gate

The targeted six-configuration standard-representation Double DQN screen failed badly; the best screen exceeded 1.2 million cost. The protected 115,819.00 public artifact was kept.

## 8. Learning and convergence evidence

The release retains baseline learning-curve figures under `results/` for PPO, DQN and the actor-critic baseline. For Neural Network SARSA and the final A2C replacement, the full official 200-episode holdout summaries are the primary final validation evidence. No unsupported convergence curve is fabricated.

In [ ]:
import matplotlib.pyplot as plt
for name in ['ppo_learning_curve.png','dqn_learning_curve.png','a2c_learning_curve.png']:
    path = PROJECT_ROOT / 'results' / name
    if path.exists():
        print(name)
        display(plt.imread(path))

## 9. Final policy and model mapping

| Technique | Policy | Model artifact |
|---|---|---|
| PPO | `submissions/ppo/policy.py` | `submissions/ppo/model.zip` |
| DQN | `submissions/dqn/policy.py` | `submissions/dqn/model.zip` |
| Neural Network SARSA | `submissions/neural_sarsa/policy.py` | `submissions/neural_sarsa/policy_state.pt` |
| A2C + Representation C | `submissions/a2c/policy.py` | `submissions/a2c/model.zip` |
| Double DQN | `submissions/double_dqn/policy.py` | `submissions/double_dqn/model.zip` |


## 10. Course-supplied policy validation

Run the supplied validator from the repository root against all five final policy files. It checks importability, the required `run_policy(observation)` signature, valid integer quantities, observation immutability, deterministic inference, forbidden direct `env.step()`/`env.reset()` calls, and local inference-time limits.

In [ ]:
# python policy_validation_tests.py submissions/ppo/policy.py --max-seconds-per-call 0.25
# python policy_validation_tests.py submissions/dqn/policy.py --max-seconds-per-call 0.25
# python policy_validation_tests.py submissions/neural_sarsa/policy.py --max-seconds-per-call 0.25
# python policy_validation_tests.py submissions/a2c/policy.py --max-seconds-per-call 0.25
# python policy_validation_tests.py submissions/double_dqn/policy.py --max-seconds-per-call 0.25

## 11. Final selection and integrity notes

The final portfolio is frozen as PPO, DQN, Neural Network SARSA, A2C + Representation C and Double DQN. The official environment, assigned V030 configuration, official cost function and evaluator are unchanged. Final policy inference uses only the documented observation and performs no training, environment stepping/resetting, future-demand access, hidden-parameter access or internet access.

Rejected tuning artifacts are intentionally excluded from the final release. The final report is concise and records implementation summary, experimental approach, results, observations and inference as required by the course instructions.